# LoRA (Low-Rank Adaptation) from Scratch — Solution

## Setup

In [ ]:
!pip install torch matplotlib --quiet

In [ ]:
import math
import copy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

---
## Stage 1 — Linear Algebra Foundations

### 1a — SVD and low-rank approximation

In [ ]:
def low_rank_approx(W: torch.Tensor, rank: int) -> torch.Tensor:
    """
    Best rank-r approximation via truncated SVD (Eckart-Young theorem).
    W = U @ diag(S) @ Vh  →  W_r = U[:,:r] @ diag(S[:r]) @ Vh[:r,:]
    """
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    # Truncate to top-r components
    U_r  = U[:, :rank]           # [d, r]
    S_r  = S[:rank]              # [r]
    Vh_r = Vh[:rank, :]          # [r, k]
    return U_r @ torch.diag(S_r) @ Vh_r

def reconstruction_error(W: torch.Tensor, W_approx: torch.Tensor) -> float:
    """Normalized Frobenius error."""
    return (W - W_approx).norm("fro").item() / W.norm("fro").item()

def explained_variance(W: torch.Tensor, rank: int) -> float:
    """
    Fraction of total spectral energy captured by top-r singular values.
    S^2 = squared singular values = eigenvalues of W^T W.
    """
    S = torch.linalg.svdvals(W)
    return (S[:rank] ** 2).sum().item() / (S ** 2).sum().item()

# Experiment
d, k = 256, 256
ranks = [1, 2, 4, 8, 16, 32, 64, 128]

W_random  = torch.randn(d, k)
r_true    = 8
W_lowrank = torch.randn(d, r_true) @ torch.randn(r_true, k) + 0.01 * torch.randn(d, k)

errors_random  = [reconstruction_error(W_random,  low_rank_approx(W_random,  r)) for r in ranks]
errors_lowrank = [reconstruction_error(W_lowrank, low_rank_approx(W_lowrank, r)) for r in ranks]
varexp_random  = [explained_variance(W_random,  r) for r in ranks]
varexp_lowrank = [explained_variance(W_lowrank, r) for r in ranks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(ranks, errors_random,  "o-", label="random W",   color="coral")
ax1.plot(ranks, errors_lowrank, "o-", label="low-rank W", color="steelblue")
ax1.set_xlabel("Approximation rank r"); ax1.set_ylabel("Relative Frobenius error")
ax1.set_title("Low-rank approximation error"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(ranks, varexp_random,  "o-", label="random W",   color="coral")
ax2.plot(ranks, varexp_lowrank, "o-", label="low-rank W", color="steelblue")
ax2.axhline(0.99, linestyle="--", color="gray", label="99% variance")
ax2.set_xlabel("Rank r"); ax2.set_ylabel("Explained variance")
ax2.set_title("Explained variance by rank"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

for r, ev_r, ev_l in zip(ranks, varexp_random, varexp_lowrank):
    print(f"r={r:>3}  random: {ev_r:.3f}  low-rank: {ev_l:.3f}")

### 1b — Singular value spectrum

In [ ]:
def plot_singular_value_spectrum(matrices: dict, title: str = "Singular value spectra"):
    fig, ax = plt.subplots(figsize=(9, 4))
    colors  = ["coral", "steelblue", "seagreen"]
    for (label, W), color in zip(matrices.items(), colors):
        S = torch.linalg.svdvals(W.float())
        S_norm = S / S[0]                         # normalize by largest SV
        ax.plot(S_norm.numpy(), label=label, color=color, linewidth=2)
    ax.set_xlabel("Singular value index"); ax.set_ylabel("Normalized singular value")
    ax.set_title(title); ax.legend(); ax.set_yscale("log"); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

def make_pretrained_weight(d, k, effective_rank=32, noise=0.01):
    signal    = torch.randn(d, effective_rank) @ torch.randn(effective_rank, k)
    noise_mat = noise * torch.randn(d, k)
    W         = signal + noise_mat
    return W / W.norm()

matrices = {
    "random (no structure)":  torch.randn(256, 256),
    "pretrained-like (r=32)": make_pretrained_weight(256, 256, effective_rank=32),
    "pretrained-like (r=8)":  make_pretrained_weight(256, 256, effective_rank=8),
}
plot_singular_value_spectrum(matrices)

### 1c — Rank of BA

In [ ]:
def verify_lora_rank(d: int, k: int, r: int):
    B = torch.randn(d, r)
    A = torch.randn(r, k)
    delta_W = B @ A
    rank_dW = torch.linalg.matrix_rank(delta_W).item()
    full_params = d * k
    lora_params = r * (d + k)
    print(f"r={r:>2}  |  delta_W shape: [{d}×{k}]  |  rank(BA)={rank_dW}  |  "
          f"LoRA params: {lora_params:,}  vs full: {full_params:,}  ({lora_params/full_params:.2%})")

for r in [1, 4, 8, 16]:
    verify_lora_rank(d=512, k=512, r=r)

---
## Stage 2 — LoRA Layer Implementation

In [ ]:
class LoRALinear(nn.Module):
    """
    Linear layer with LoRA bypass.
    h = Wx + (alpha/r) * B A x
    """
    def __init__(
        self,
        in_features:  int,
        out_features: int,
        rank:         int   = 4,
        alpha:        float = None,
        bias:         bool  = True,
    ):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.rank         = rank
        self.alpha        = alpha if alpha is not None else float(rank)
        self.scaling      = self.alpha / self.rank
        self.merged       = False

        # Frozen pretrained weight — requires_grad=False
        self.weight = nn.Parameter(
            torch.empty(out_features, in_features), requires_grad=False
        )
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter("bias", None)

        # Trainable LoRA matrices
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, rank))

        self.reset_lora_parameters()

    def reset_lora_parameters(self):
        """
        A ~ kaiming_uniform, B = 0.
        Guarantees delta_W = BA = 0 at init → model starts at pretrained checkpoint.
        """
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = F.linear(x, self.weight, self.bias)
        if self.merged:
            return base
        # LoRA path: x → A → B, scaled by alpha/r
        lora = F.linear(F.linear(x, self.lora_A), self.lora_B)
        return base + self.scaling * lora

    def merge_weights(self):
        """W ← W + scaling × B @ A  (in-place, zero inference overhead)."""
        if not self.merged:
            self.weight.data += self.scaling * (self.lora_B @ self.lora_A)
            self.merged = True

    def unmerge_weights(self):
        """Undo merge — useful to resume training."""
        if self.merged:
            self.weight.data -= self.scaling * (self.lora_B @ self.lora_A)
            self.merged = False

    def trainable_parameters(self) -> int:
        return self.lora_A.numel() + self.lora_B.numel()

    def extra_repr(self) -> str:
        return (f"in={self.in_features}, out={self.out_features}, "
                f"r={self.rank}, alpha={self.alpha}, merged={self.merged}")

In [ ]:
# Correctness checks
layer = LoRALinear(in_features=64, out_features=32, rank=4, alpha=4)
x     = torch.randn(8, 64)

delta_W = layer.scaling * layer.lora_B @ layer.lora_A
assert delta_W.abs().max().item() == 0.0
print("✓ delta_W = 0 at initialization")

base_out = F.linear(x, layer.weight, layer.bias)
lora_out = layer(x)
assert torch.allclose(base_out, lora_out, atol=1e-6)
print("✓ LoRA output matches base at initialization")

with torch.no_grad():
    layer.lora_B.data = torch.randn_like(layer.lora_B)
assert not torch.allclose(base_out, layer(x))
print("✓ LoRA output changes after B is updated")

out_before = layer(x).clone()
layer.merge_weights()
assert torch.allclose(out_before, layer(x), atol=1e-5)
print("✓ merge_weights() preserves output")

layer.unmerge_weights()
assert torch.allclose(out_before, layer(x), atol=1e-5)
print("✓ unmerge_weights() restores original behavior")

total  = layer.weight.numel() + layer.bias.numel()
lora_p = layer.trainable_parameters()
print(f"\nFull: {total:,}  LoRA trainable: {lora_p:,}  ({lora_p/total:.2%})")

trainable = [p for p in layer.parameters() if p.requires_grad]
assert layer.weight not in trainable
print("✓ W is frozen")

### 2b — Parameter count table

In [ ]:
def lora_param_efficiency(
    d_model:        int  = 4096,
    ranks:          list = [1, 2, 4, 8, 16, 32, 64],
    n_layers:       int  = 32,
    target_modules: int  = 4,
):
    full_per_layer = d_model * d_model
    full_total     = n_layers * target_modules * full_per_layer

    print(f"Model: d={d_model}, {n_layers} layers, {target_modules} target modules/layer")
    print(f"Full fine-tune: {full_total/1e6:.0f}M params\n")
    print(f"{'r':>4}  {'LoRA params':>14}  {'Ratio':>8}  {'Params/layer':>14}")
    print("-" * 46)
    for r in ranks:
        lora_per_layer = r * (d_model + d_model)     # A + B
        lora_total     = n_layers * target_modules * lora_per_layer
        ratio          = lora_total / full_total
        print(f"{r:>4}  {lora_total/1e6:>13.2f}M  {ratio:>8.3%}  {lora_per_layer/1e3:>12.1f}K")

lora_param_efficiency()

---
## Stage 3 — Inject LoRA into a Transformer

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int   = 256
    seq_len:    int   = 64
    n_layers:   int   = 4
    d_model:    int   = 128
    n_heads:    int   = 4
    d_ff:       int   = 512
    dropout:    float = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg, linear_cls=nn.Linear):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.d_head  = cfg.d_model // cfg.n_heads
        self.q_proj  = linear_cls(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj  = linear_cls(cfg.d_model, cfg.d_model, bias=False)
        self.v_proj  = linear_cls(cfg.d_model, cfg.d_model, bias=False)
        self.o_proj  = linear_cls(cfg.d_model, cfg.d_model, bias=False)
        self.drop    = nn.Dropout(cfg.dropout)
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(cfg.seq_len, cfg.seq_len)).view(1,1,cfg.seq_len,cfg.seq_len)
        )

    def forward(self, x):
        B, T, C = x.shape
        def reshape(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = reshape(self.q_proj(x)), reshape(self.k_proj(x)), reshape(self.v_proj(x))
        att = (q @ k.transpose(-2,-1)) / math.sqrt(self.d_head)
        att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float("-inf"))
        att = self.drop(torch.softmax(att, dim=-1))
        out = (att @ v).transpose(1,2).contiguous().view(B, T, C)
        return self.o_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, cfg, linear_cls=nn.Linear):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg, linear_cls)
        self.ln2  = nn.LayerNorm(cfg.d_model)
        self.ff   = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, cfg, linear_cls=nn.Linear):
        super().__init__()
        self.embed  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos    = nn.Embedding(cfg.seq_len, cfg.d_model)
        self.blocks = nn.ModuleList([TransformerBlock(cfg, linear_cls) for _ in range(cfg.n_layers)])
        self.ln_f   = nn.LayerNorm(cfg.d_model)
        self.head   = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        x = self.embed(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))

    def count_params(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable

cfg        = GPTConfig()
base_model = TinyGPT(cfg).to(device)

In [ ]:
def inject_lora(
    model:          nn.Module,
    rank:           int   = 4,
    alpha:          float = None,
    target_modules: list  = ["q_proj", "v_proj"],
) -> nn.Module:
    """
    Replace target nn.Linear modules with LoRALinear in-place.
    """
    # Step 1: freeze everything
    for p in model.parameters():
        p.requires_grad = False

    # Step 2: replace target linear layers
    for full_name, module in list(model.named_modules()):
        # Check if this module's leaf name matches a target
        leaf_name = full_name.split(".")[-1]
        if not isinstance(module, nn.Linear) or leaf_name not in target_modules:
            continue

        # Build the LoRALinear replacement
        lora_layer = LoRALinear(
            in_features  = module.in_features,
            out_features = module.out_features,
            rank         = rank,
            alpha        = alpha,
            bias         = module.bias is not None,
        )
        # Copy pretrained weight over (keep it frozen)
        with torch.no_grad():
            lora_layer.weight.copy_(module.weight)
            if module.bias is not None:
                lora_layer.bias.data.copy_(module.bias.data)

        # Navigate to parent and replace child
        parts      = full_name.split(".")
        parent     = model.get_submodule(".".join(parts[:-1])) if len(parts) > 1 else model
        setattr(parent, parts[-1], lora_layer)

    # Step 3: unfreeze LoRA params
    for module in model.modules():
        if isinstance(module, LoRALinear):
            module.lora_A.requires_grad = True
            module.lora_B.requires_grad = True

    return model

lora_model = copy.deepcopy(base_model)
lora_model = inject_lora(lora_model, rank=4, target_modules=["q_proj", "v_proj"])

total, trainable = lora_model.count_params()
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")
print(f"Trainable ratio:  {trainable/total:.2%}\n")
for name, p in lora_model.named_parameters():
    if p.requires_grad:
        print(f"  trainable: {name}  {tuple(p.shape)}")

### 3b — Training loop

In [ ]:
def make_copy_batch(batch_size, seq_len, vocab_size, device):
    tokens = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    return tokens, tokens

def train_loop(
    model,
    n_steps:    int   = 500,
    batch_size: int   = 16,
    lr:         float = 1e-3,
    label:      str   = "model",
) -> dict:
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr)
    loss_fn   = nn.CrossEntropyLoss()
    model.train()

    stats = defaultdict(list)
    for step in range(n_steps):
        x, y = make_copy_batch(batch_size, cfg.seq_len, cfg.vocab_size, device)

        logits = model(x)                                          # [B, T, V]
        loss   = loss_fn(logits.view(-1, cfg.vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()

        # Gradient norm across all trainable params
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1e9).item()

        optimizer.step()

        stats["losses"].append(loss.item())
        stats["grad_norms"].append(grad_norm)

        # Track ||delta_W||_F for each LoRA layer
        dw_norms = []
        for m in model.modules():
            if isinstance(m, LoRALinear):
                dw = (m.scaling * m.lora_B @ m.lora_A).norm().item()
                dw_norms.append(dw)
        if dw_norms:
            stats["delta_W_norms"].append(np.mean(dw_norms))

        if (step + 1) % 100 == 0:
            print(f"[{label}] step {step+1:>4}  loss={loss.item():.4f}  grad_norm={grad_norm:.3f}")

    return dict(stats)

In [ ]:
full_model = copy.deepcopy(base_model)
for p in full_model.parameters():
    p.requires_grad = True

print("Training full fine-tune...")
full_stats = train_loop(full_model, n_steps=500, lr=1e-3, label="full")

print("\nTraining LoRA...")
lora_model = copy.deepcopy(base_model)
lora_model = inject_lora(lora_model, rank=4, target_modules=["q_proj", "v_proj"])
lora_stats = train_loop(lora_model, n_steps=500, lr=1e-3, label="lora")

In [ ]:
def smooth(vals, w=20):
    return np.convolve(vals, np.ones(w)/w, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Loss curves
axes[0].plot(smooth(full_stats["losses"]), label="Full fine-tune", color="coral")
axes[0].plot(smooth(lora_stats["losses"]), label="LoRA (r=4)",    color="steelblue")
axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Panel 2: Gradient norms
axes[1].plot(smooth(full_stats["grad_norms"]), label="Full",  color="coral")
axes[1].plot(smooth(lora_stats["grad_norms"]), label="LoRA",  color="steelblue")
axes[1].set_xlabel("Step"); axes[1].set_ylabel("Gradient norm")
axes[1].set_title("Gradient norms"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Panel 3: ||delta_W|| growth (LoRA only)
if lora_stats.get("delta_W_norms"):
    axes[2].plot(lora_stats["delta_W_norms"], color="steelblue")
    axes[2].set_xlabel("Step"); axes[2].set_ylabel("||delta_W||_F")
    axes[2].set_title("LoRA update magnitude growth")
    axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## Stage 4 — Training Dynamics

### 4a — B=0 initialization matters

In [ ]:
class LoRALinearBadInit(LoRALinear):
    """Random B init — demonstrates why B=0 is necessary."""
    def reset_lora_parameters(self):
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.lora_B, a=math.sqrt(5))  # not zero!

def run_init_experiment() -> dict:
    results = {}
    for name, lora_cls in [("B=0 (correct)", LoRALinear), ("B=random (bad)", LoRALinearBadInit)]:
        m = copy.deepcopy(base_model)
        # Freeze base, inject LoRA with given cls
        for p in m.parameters():
            p.requires_grad = False
        for full_name, module in list(m.named_modules()):
            leaf = full_name.split(".")[-1]
            if isinstance(module, nn.Linear) and leaf in ["q_proj", "v_proj"]:
                ll = lora_cls(module.in_features, module.out_features, rank=4, bias=False)
                with torch.no_grad():
                    ll.weight.copy_(module.weight)
                parts  = full_name.split(".")
                parent = m.get_submodule(".".join(parts[:-1])) if len(parts) > 1 else m
                setattr(parent, parts[-1], ll)
        for mod in m.modules():
            if isinstance(mod, LoRALinear):
                mod.lora_A.requires_grad = True
                mod.lora_B.requires_grad = True

        stats = train_loop(m, n_steps=200, lr=1e-3, label=name)
        results[name] = stats

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, stats in results.items():
        color = "steelblue" if "correct" in name else "coral"
        axes[0].plot(smooth(stats["losses"]),     label=name, color=color)
        axes[1].plot(smooth(stats["grad_norms"]), label=name, color=color)
    axes[0].set_title("Loss: init comparison");       axes[0].set_xlabel("Step"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].set_title("Grad norm: init comparison");  axes[1].set_xlabel("Step"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    return results

init_results = run_init_experiment()

### 4b — Rank sensitivity

In [ ]:
def rank_sensitivity_experiment(
    ranks:   list = [1, 2, 4, 8, 16, 32],
    n_steps: int  = 300,
) -> dict:
    results = {}
    for r in ranks:
        m = copy.deepcopy(base_model)
        m = inject_lora(m, rank=r, target_modules=["q_proj", "v_proj"])
        stats = train_loop(m, n_steps=n_steps, lr=1e-3, label=f"r={r}")

        # Measure effective rank of learned delta_W
        eff_ranks = []
        for mod in m.modules():
            if isinstance(mod, LoRALinear):
                dw       = (mod.scaling * mod.lora_B @ mod.lora_A).detach()
                eff_rank = torch.linalg.matrix_rank(dw, atol=1e-3).item()
                eff_ranks.append(eff_rank)

        results[r] = {
            "final_loss":       stats["losses"][-1],
            "n_trainable":      sum(p.numel() for p in m.parameters() if p.requires_grad),
            "mean_eff_rank":    np.mean(eff_ranks),
            "losses":           stats["losses"],
        }
        print(f"r={r:>2}  loss={results[r]['final_loss']:.4f}  eff_rank={results[r]['mean_eff_rank']:.1f}")
    return results

rank_results = rank_sensitivity_experiment()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rs          = list(rank_results.keys())
final_losses = [rank_results[r]["final_loss"] for r in rs]
eff_ranks    = [rank_results[r]["mean_eff_rank"] for r in rs]

axes[0].plot(rs, final_losses, "o-", color="steelblue")
axes[0].set_xlabel("Rank r"); axes[0].set_ylabel("Final loss")
axes[0].set_title("Loss vs rank"); axes[0].grid(True, alpha=0.3)

axes[1].plot(rs, eff_ranks, "o-", color="coral", label="effective rank")
axes[1].plot(rs, rs, "--", color="gray", label="allocated rank")
axes[1].set_xlabel("Allocated rank r"); axes[1].set_ylabel("Effective rank of delta_W")
axes[1].set_title("Effective vs allocated rank")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

colors_r = plt.cm.viridis(np.linspace(0, 1, len(rs)))
for r, color in zip(rs, colors_r):
    axes[2].plot(smooth(rank_results[r]["losses"]), label=f"r={r}", color=color)
axes[2].set_xlabel("Step"); axes[2].set_ylabel("Loss")
axes[2].set_title("Learning curves by rank"); axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

### 4c — Alpha / scaling sensitivity

In [ ]:
def alpha_sensitivity_experiment(
    rank:    int   = 8,
    alphas:  list  = [0.5, 1, 2, 4, 8, 16, 32],
    n_steps: int   = 300,
    lr:      float = 1e-3,
) -> dict:
    results = {}
    for alpha in alphas:
        m = copy.deepcopy(base_model)
        m = inject_lora(m, rank=rank, alpha=alpha, target_modules=["q_proj", "v_proj"])
        stats = train_loop(m, n_steps=n_steps, lr=lr, label=f"α={alpha}")

        dw_norms = []
        for mod in m.modules():
            if isinstance(mod, LoRALinear):
                dw_norms.append((mod.scaling * mod.lora_B @ mod.lora_A).norm().item())

        results[alpha] = {
            "final_loss":    stats["losses"][-1],
            "mean_dW_norm":  np.mean(dw_norms),
            "scaling":       alpha / rank,
            "losses":        stats["losses"],
        }
    return results

alpha_results = alpha_sensitivity_experiment()

scalings     = [v["scaling"]     for v in alpha_results.values()]
final_losses = [v["final_loss"]  for v in alpha_results.values()]
dw_norms     = [v["mean_dW_norm"] for v in alpha_results.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(scalings, final_losses, "o-", color="steelblue")
ax1.set_xlabel("Scaling (alpha/r)"); ax1.set_ylabel("Final loss")
ax1.set_title("Loss vs scaling factor"); ax1.set_xscale("log"); ax1.grid(True, alpha=0.3)

ax2.plot(scalings, dw_norms, "o-", color="coral")
ax2.set_xlabel("Scaling (alpha/r)"); ax2.set_ylabel("||delta_W||_F")
ax2.set_title("Update magnitude vs scaling"); ax2.set_xscale("log"); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("Key insight: alpha acts as a multiplier on effective LR for the LoRA update.")
print("Too small → update too weak. Too large → unstable / overpowers pretrained weights.")

### 4d — Intrinsic rank of learned delta_W

In [ ]:
def analyze_learned_delta_W(model: nn.Module) -> None:
    lora_layers = [(n, m) for n, m in model.named_modules() if isinstance(m, LoRALinear)]
    n_layers    = len(lora_layers)
    if n_layers == 0:
        print("No LoRA layers found.")
        return

    fig, axes = plt.subplots(1, n_layers, figsize=(4 * n_layers, 4))
    if n_layers == 1:
        axes = [axes]

    for ax, (name, m) in zip(axes, lora_layers):
        dw   = (m.scaling * m.lora_B @ m.lora_A).detach()
        S    = torch.linalg.svdvals(dw.float())
        S_n  = (S / S[0]).numpy()
        eff  = torch.linalg.matrix_rank(dw, atol=1e-3).item()

        ax.bar(range(len(S_n)), S_n, color="steelblue", alpha=0.8)
        ax.set_xlabel("Singular value index")
        ax.set_ylabel("Normalized magnitude")
        ax.set_title(f"{name.split('.')[-1]}\neff rank={eff} / r={m.rank}")
        ax.set_yscale("log"); ax.grid(True, alpha=0.3)

    plt.suptitle("Singular value spectrum of learned delta_W = scaling × BA", y=1.02)
    plt.tight_layout(); plt.show()

analyze_learned_delta_W(lora_model)

---
## Stage 5 — Weight Merging and Inference

In [ ]:
def merge_all_lora_layers(model: nn.Module) -> nn.Module:
    for m in model.modules():
        if isinstance(m, LoRALinear):
            m.merge_weights()
    return model

def convert_lora_to_linear(model: nn.Module) -> nn.Module:
    """Merge + replace LoRALinear → nn.Linear for zero LoRA overhead."""
    merge_all_lora_layers(model)

    for full_name, module in list(model.named_modules()):
        if not isinstance(module, LoRALinear):
            continue
        new_linear = nn.Linear(
            module.in_features, module.out_features,
            bias=(module.bias is not None)
        )
        with torch.no_grad():
            new_linear.weight.copy_(module.weight)
            if module.bias is not None:
                new_linear.bias.copy_(module.bias)
        parts  = full_name.split(".")
        parent = model.get_submodule(".".join(parts[:-1])) if len(parts) > 1 else model
        setattr(parent, parts[-1], new_linear)

    return model

In [ ]:
x_test     = torch.randint(0, cfg.vocab_size, (4, cfg.seq_len), device=device)
lora_model.eval()
out_before = lora_model(x_test).detach()

merged_model = copy.deepcopy(lora_model)
merged_model = merge_all_lora_layers(merged_model)
assert torch.allclose(out_before, merged_model(x_test).detach(), atol=1e-4)
print("✓ Merged model outputs match")

converted_model = convert_lora_to_linear(copy.deepcopy(lora_model))
assert torch.allclose(out_before, converted_model(x_test).detach(), atol=1e-4)
print("✓ Converted model outputs match")

# Inference speed comparison
models_to_bench = {
    "LoRA (unmerged)": lora_model,
    "LoRA (merged)":   merged_model,
    "Plain Linear":    converted_model,
}
n_warmup, n_bench = 10, 100
print(f"\n{'Model':<22}  {'ms/pass':>10}")
print("-" * 36)
for name, m in models_to_bench.items():
    m.eval()
    with torch.no_grad():
        for _ in range(n_warmup): m(x_test)
        t0 = time.time()
        for _ in range(n_bench):  m(x_test)
        ms = (time.time() - t0) / n_bench * 1000
    print(f"{name:<22}  {ms:>10.3f}")

In [ ]:
def visualize_lora_updates(model: nn.Module, max_layers: int = 4) -> None:
    lora_layers = [(n, m) for n, m in model.named_modules() if isinstance(m, LoRALinear)]
    lora_layers = lora_layers[:max_layers]
    if not lora_layers:
        return

    fig, axes = plt.subplots(len(lora_layers), 2, figsize=(10, 3 * len(lora_layers)))
    if len(lora_layers) == 1:
        axes = [axes]

    for row, (name, m) in enumerate(lora_layers):
        W  = m.weight.detach().cpu().float()
        dW = (m.scaling * m.lora_B @ m.lora_A).detach().cpu().float()
        ratio = dW.norm().item() / W.norm().item()

        vmax_W  = W.abs().quantile(0.99).item()
        vmax_dW = dW.abs().quantile(0.99).item() if dW.abs().max() > 1e-8 else 0.1

        im0 = axes[row][0].imshow(W.numpy(),  cmap="RdBu_r", vmin=-vmax_W,  vmax=vmax_W,  aspect="auto")
        im1 = axes[row][1].imshow(dW.numpy(), cmap="RdBu_r", vmin=-vmax_dW, vmax=vmax_dW, aspect="auto")
        axes[row][0].set_title(f"{name.split('.')[-1]} — W")
        axes[row][1].set_title(f"delta_W  (||dW||/||W|| = {ratio:.3f})")
        plt.colorbar(im0, ax=axes[row][0], fraction=0.03)
        plt.colorbar(im1, ax=axes[row][1], fraction=0.03)

    plt.suptitle("W vs delta_W heatmaps", y=1.01)
    plt.tight_layout(); plt.show()

visualize_lora_updates(lora_model)

---
## Appendix — Why the Scaling Works

The LoRA paper uses `scaling = alpha / r`. This is more subtle than it looks.

**Without scaling** (`scaling = 1`):
When you increase `r`, more parameters → larger initial gradient signal → effectively larger learning rate for the LoRA path. Changing `r` forces you to retune `lr`.

**With `alpha / r` scaling**:
The effective learning rate for the LoRA update becomes proportional to `lr × alpha / r`.
Setting `alpha = r` makes scaling = 1. Setting `alpha = 2r` doubles the effective LoRA lr.

This decouples `r` from `lr` — you can sweep ranks without retuning the optimizer.

**In practice**: The original paper uses `alpha = r` (scaling=1) or `alpha = 2r`.
For large models, `alpha = 16` with various `r` works well as a default.

---

## Key Numbers to Know (LLaMA-7B)

| | Full fine-tune | LoRA (r=8) | LoRA (r=16) |
|---|---|---|---|
| Trainable params | 7B | ~4M | ~8M |
| % of total | 100% | 0.06% | 0.12% |
| GPU memory | ~112 GB | ~14 GB | ~14 GB |
| Convergence | slow | fast | fast |


---
## Extension Exercises

**1. QLoRA**
Quantize W to int8, keep A and B in fp32. Measure memory vs accuracy.

**2. AdaLoRA**
Periodically SVD delta_W and prune smallest singular values.
Redistribute rank budget from unimportant layers to important ones.

**3. LoRA for feed-forward layers**
Apply to up/down projections. Compare efficiency vs attention-only.

**4. Multi-task merging**
W' = W + alpha1*B1A1 + alpha2*B2A2. Show adapters can be linearly combined.

**5. Rank collapse analysis**
Train r=32 on a simple task. Plot singular value spectrum at checkpoints 0/100/300/500.
Observe how many SVs survive — that's the intrinsic rank of the task.